In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-lgd-sensitivity-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import os
import pandas as pd
import numpy as np
import sys
import catboost as cb
import sklearn.metrics as skm

# change wd to absolute path
os.chdir('/tmp')

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# lambda handler
def lambda_handler(event, context):
    # get string of column to drop
    str_col = event['feature']
    print(f'Column to drop: {str_col}')
    
    # constants
    str_project = '20231010-gen-xii'
    str_target = 'target'
    
    ###############################################################################
    # HYPERPARAMETERS
    ###############################################################################
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))

    # get number of iterations
    int_n_iterations = int(dict_hyperparameters['INT_N_ITERATIONS'])
    print(f'Iterations: {int_n_iterations}')

    # get filename for training
    str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
    print(f'Training filename: {str_filename_train}')

    # get filename for valid
    str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
    print(f'Valid filename: {str_filename_valid}')

    # get proportion of iterations to use as early stopping rounds
    flt_prop_early_stopping = float(dict_hyperparameters['PROP_EARLY_STOPPING'])
    print(f'Proportion early stopping: {flt_prop_early_stopping}')

    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')

    ##################################################################################

    # get list of features
    print('Importing list_of_starting_features...')
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_model = list(pd.read_csv(str_uri)['feature'])

    # rm str_col
    print(f'Removing {str_col} from model...')
    list_cols_model = [col for col in list_cols_model if col != str_col]
    print(f'After dropping {str_col}, there will be {len(list_cols_model)} features in the model')
    # append target
    list_cols_import = list_cols_model + [str_target]

    # get lr
    print('Getting learning rate...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df_tuning = pd.read_csv(str_uri)
    # get learning rate
    flt_learning_rate = df_tuning['learning_rate'].iloc[0]
    print(f'Learning rate: {flt_learning_rate}')

    # read training data
    print('Reading training data...')
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get the non numeric feats
    print('Getting list of non-numeric columns...')
    list_cols_non_numeric = []
    for col in list_cols_model:
        if df[col].dtype not in ['float64','int64']:
            list_cols_non_numeric.append(col)

    # build model
    print('Building model...')
    # pool data
    pool_train = cb.Pool(
        df[list_cols_model], 
        df[str_target], 
        cat_features=list_cols_non_numeric,
    )
    del df

    # read validation data
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # pool data
    pool_valid = cb.Pool(
        df[list_cols_model], 
        df[str_target], 
        cat_features=list_cols_non_numeric,
    )
    del df

    # constraints
    dict_monotone_constraints = {
        # better
        'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
        'fltapproveddowntotal__app': -1,
        'fltdowncash__app': -1,
        'bookvalue__app': -1,
        'ENG-dealership_age': -1,
        'bookvalue__app': -1,
        'fltdowncash__app': -1,
        'fltapproveddowntotal__app': -1,
        # worse
        'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
        'ENG-loan_to_value': 1,
        'ENG-payment_to_income': 1,
        'ENG-vehicle_age': 1,
        'fltadvance__app': 1,
        'bigmileage_odometer__app': 1,
        'amtfinanced__app': 1,
        'miles_odometer__app': 1,
        'pti__app': 1,
        'fltadvance__app': 1,
        'ENG-loan_to_value': 1,
        'amtfinanced__app': 1,
    }
    # ensure features are in list_cols_model
    dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

    # init class
    cls_model_inference = cb.CatBoostRegressor(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric=str_eval_metric,
        iterations=int_n_iterations,
        learning_rate=flt_learning_rate,
        monotone_constraints=dict_monotone_constraints,
    )

    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=100,
        use_best_model=True,
        early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
    )
    del pool_train
    del pool_valid

    ################################################################################################
    # GET TRAINING EVAL METRIC
    ################################################################################################
    print('Getting training eval metric...')

    # import data
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get predictions
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])

    # get eval metric - train
    if str_eval_metric == 'RMSE':
        flt_eval_metric_train = np.sqrt(skm.mean_squared_error(y_true=df[str_target], y_pred=df['y_hat']))
    else:
        pass

    # save memory
    del df

    ################################################################################################
    # GET VALIDATION EVAL METRIC (WEIGHTED)
    ################################################################################################
    print('Getting validation eval metric...')

    # import data
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get predictions
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])

    # get eval metric - valid
    if str_eval_metric == 'RMSE':
        flt_eval_metric_valid = np.sqrt(skm.mean_squared_error(y_true=df[str_target], y_pred=df['y_hat']))
    else:
        pass

    # save memory
    del df

    ################################################################################################
    # CREATE OUTPUT DATA FRAME
    ################################################################################################
    print('Creating output data frame...')

    flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
    dict_row = {
        'feature': str_col,
        'learning_rate': flt_learning_rate,
        'flt_eval_metric_train': flt_eval_metric_train,
        'flt_eval_metric_valid': flt_eval_metric_valid,
        'diff': flt_diff,
        'best_iteration': cls_model_inference.get_best_iteration(),
    }
    df = pd.DataFrame(dict_row, index=[0])
    # save
    str_filename = f'df_output_{str_col}.csv'
    str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/04_batch_sensitivity_analysis/models/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-lgd-sensitivity-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  113.2kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> a00d7145cc1d
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 2b238a4283cb
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> bc4b4040c4c5
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 0655e38a46de
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 994a638f0e96
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 28821caf1432
Removing intermediate container 28821caf1432
 ---> 0da641ed8df2
Successfully built 0da641ed8df2
Successfully tagged genxii-lgd-sensitivity-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-sensitivity-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-sensitivity-2]
7d880db39e1c: Preparing
7274ff74e3e0: Preparing
c621ed74adab: Preparing
dbd3c2db499f: Preparing
45d114d35c45: Preparing
86e4d644d316: Preparing
15dd6c63f3a2: Preparing
dd00ec5a5244: Preparing
8f43d000b361: Preparing
c349004e3af3: Preparing
15dd6c63f3a2: Waiting
86e4d644d316: Waiting
8f43d000b361: Waiting
dbd3c2db499f: Layer already exists
45d114d35c45: Layer already exists
c621ed74adab: Layer already exists
7274ff74e3e0: Layer already exists
86e4d644d316: Layer already exists
dd00ec5a5244: Layer already exists
15dd6c63f3a2: Layer already exists
8f43d000b361: Layer already exists
c349004e3af3: Layer already exists
7d880db39e1c: Pushed
latest: digest: sha256:c17b82776eb6f3fde194cdea14f319632fbfd87678f90b7f65b6ed586bc654c9 size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 08 Nov 2023 22:52:39 GMT',
                                      'x-amzn-requestid': '7d28508d-897b-4d57-86d9-d4a8e758227f'},
                      'HTTPStatusCode': 204,
                      'RequestId': '7d28508d-897b-4d57-86d9-d4a8e758227f',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900,
    MemorySize=1000,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'c17b82776eb6f3fde194cdea14f319632fbfd87678f90b7f65b6ed586bc654c9',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-sensitivity-2',
 'FunctionName': 'genxii-lgd-sensitivity-2',
 'LastModified': '2023-11-08T22:52:39.945+0000',
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1071',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 08 Nov 2023 22:52:40 GMT',
                                      'x-amzn-requestid': 'b6433170-7ece-42be-9cd6-07ae3f10de70'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'b6433170-7ece-42be-9cd6-07ae3f10de70',
                      'RetryAttempts': 0},
 'RevisionId': 'a040f698-337e-45cd-865

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)